# 💸 US Wages vs Inflation by Sector (2000–2026)
**Analyzing whether pay has kept up with the cost of living across Retail, Healthcare, Tech, and Education.**

Data sourced from Bureau of Labor Statistics (BLS), FRED St. Louis Fed, and Atlanta Fed Wage Growth Tracker.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='darkgrid')

SECTOR_COLORS = {
    'Retail':      '#E74C3C',
    'Healthcare':  '#2ECC71',
    'Tech':        '#3498DB',
    'Education':   '#F39C12'
}

print('✅ Libraries loaded successfully')

## 2. Load & Explore the Data

In [ ]:
df = pd.read_csv('wages_inflation_data.csv')

# Cumulative wage growth vs cumulative inflation since 2000
df['cum_cpi'] = (1 + df['cpi_annual_pct'] / 100).cumprod() * 100
df['cum_retail'] = (1 + df['retail_wage_growth'] / 100).cumprod() * 100
df['cum_healthcare'] = (1 + df['healthcare_wage_growth'] / 100).cumprod() * 100
df['cum_tech'] = (1 + df['tech_wage_growth'] / 100).cumprod() * 100
df['cum_education'] = (1 + df['education_wage_growth'] / 100).cumprod() * 100

print(f'Years covered: {df["year"].min()} – {df["year"].max()}')
print(f'Total CPI increase 2000–2026: {df["cum_cpi"].iloc[-1] - 100:.1f}%')
print(f'\nCumulative wage growth by sector 2000–2026:')
print(f'  Retail:     {df["cum_retail"].iloc[-1] - 100:.1f}%')
print(f'  Healthcare: {df["cum_healthcare"].iloc[-1] - 100:.1f}%')
print(f'  Tech:       {df["cum_tech"].iloc[-1] - 100:.1f}%')
print(f'  Education:  {df["cum_education"].iloc[-1] - 100:.1f}%')
print(f'\nInflation outpaced wages in these sectors:')
for sector in ['retail', 'healthcare', 'tech', 'education']:
    gap = df[f'cum_{sector}'].iloc[-1] - df['cum_cpi'].iloc[-1]
    status = '✅ Ahead' if gap > 0 else '❌ Behind'
    print(f'  {sector.capitalize()}: {gap:+.1f}% vs inflation  {status}')

## 3. Annual Wage Growth vs Inflation — All Sectors

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

# Inflation as shaded area
ax.fill_between(df['year'], df['cpi_annual_pct'], alpha=0.15, color='gray', label='_nolegend_')
ax.plot(df['year'], df['cpi_annual_pct'], 'k--', linewidth=2.5, label='CPI Inflation', zorder=5)

for sector, col in [('Retail', 'retail_wage_growth'), ('Healthcare', 'healthcare_wage_growth'),
                     ('Tech', 'tech_wage_growth'), ('Education', 'education_wage_growth')]:
    ax.plot(df['year'], df[col], 'o-', color=SECTOR_COLORS[sector],
            linewidth=2, markersize=5, label=sector)

# Annotate key events
ax.axvspan(2007.5, 2009.5, alpha=0.08, color='red')
ax.text(2008.5, 8.5, 'GFC', ha='center', fontsize=9, color='red', fontweight='bold')
ax.axvspan(2019.8, 2020.8, alpha=0.08, color='purple')
ax.text(2020.3, 8.5, 'COVID', ha='center', fontsize=9, color='purple', fontweight='bold')
ax.axvspan(2021.5, 2023, alpha=0.08, color='orange')
ax.text(2022.2, 8.5, 'Inflation\nSpike', ha='center', fontsize=8, color='darkorange', fontweight='bold')

ax.set_title('Annual Wage Growth vs CPI Inflation by Sector (2000–2026)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Annual % Change', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.legend(fontsize=11, loc='upper left')
ax.set_xticks(df['year'][::2])

plt.tight_layout()
plt.savefig('01_wage_growth_vs_inflation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 4. Cumulative Growth — Are Workers Actually Ahead or Behind?

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

ax.fill_between(df['year'], df['cum_cpi'], 100, alpha=0.12, color='gray')
ax.plot(df['year'], df['cum_cpi'], 'k--', linewidth=3, label='Cumulative Inflation (CPI)', zorder=6)

for sector, col in [('Retail', 'cum_retail'), ('Healthcare', 'cum_healthcare'),
                     ('Tech', 'cum_tech'), ('Education', 'cum_education')]:
    ax.plot(df['year'], df[col], 'o-', color=SECTOR_COLORS[sector],
            linewidth=2.5, markersize=6, label=sector)
    # End point label
    ax.annotate(f"{df[col].iloc[-1] - 100:.0f}%",
                (df['year'].iloc[-1], df[col].iloc[-1]),
                textcoords='offset points', xytext=(8, 0),
                fontsize=10, color=SECTOR_COLORS[sector], fontweight='bold')

ax.annotate(f"{df['cum_cpi'].iloc[-1] - 100:.0f}%",
            (df['year'].iloc[-1], df['cum_cpi'].iloc[-1]),
            textcoords='offset points', xytext=(8, 0),
            fontsize=10, color='black', fontweight='bold')

ax.axhline(100, color='gray', linewidth=0.8, linestyle=':')
ax.text(2000.3, 101, 'Baseline (2000)', fontsize=8, color='gray')

ax.set_title('Cumulative Wage Growth vs Inflation Since 2000\n(100 = starting point in 2000)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Cumulative Index (2000 = 100)', fontsize=12)
ax.legend(fontsize=11, loc='upper left')
ax.set_xticks(df['year'][::2])

plt.tight_layout()
plt.savefig('02_cumulative_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5. Real Wage Growth — What Workers Actually Gained or Lost Each Year

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

sectors = [
    ('Retail', 'retail_real_wage'),
    ('Healthcare', 'healthcare_real_wage'),
    ('Tech', 'tech_real_wage'),
    ('Education', 'education_real_wage')
]

for i, (sector, col) in enumerate(sectors):
    ax = axes[i]
    color = SECTOR_COLORS[sector]
    bar_colors = ['#E74C3C' if v < 0 else color for v in df[col]]
    
    bars = ax.bar(df['year'], df[col], color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
    
    # Shade COVID and inflation spike
    ax.axvspan(2019.5, 2020.5, alpha=0.1, color='purple')
    ax.axvspan(2021.5, 2023.5, alpha=0.1, color='orange')
    
    years_negative = df[df[col] < 0]['year'].tolist()
    neg_count = len(years_negative)
    
    ax.set_title(f'{sector} — Real Wage Growth\n({neg_count} years of falling real wages)',
                 fontweight='bold', fontsize=11)
    ax.set_ylabel('Real Wage Growth (%)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
    ax.set_xticks(df['year'][::4])
    ax.tick_params(axis='x', rotation=30)

# Shared legend
positive_patch = mpatches.Patch(color='steelblue', label='Real wage gain (wages > inflation)')
negative_patch = mpatches.Patch(color='#E74C3C', label='Real wage loss (inflation > wages)')
fig.legend(handles=[positive_patch, negative_patch], loc='lower center',
           ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Real Wage Growth by Sector (2000–2026)\nRed bars = inflation exceeded wage growth that year',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_real_wage_by_sector.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 6. The Inflation Spike of 2022 — Who Got Hit Hardest?

In [ ]:
# Focus on 2019-2024: the COVID and inflation spike window
crisis = df[(df['year'] >= 2019) & (df['year'] <= 2024)].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: wage vs inflation gap in 2022 (worst year)
year_2022 = df[df['year'] == 2022].iloc[0]
sectors_2022 = ['Retail', 'Healthcare', 'Tech', 'Education']
wage_2022 = [year_2022['retail_wage_growth'], year_2022['healthcare_wage_growth'],
             year_2022['tech_wage_growth'], year_2022['education_wage_growth']]
real_2022 = [year_2022['retail_real_wage'], year_2022['healthcare_real_wage'],
             year_2022['tech_real_wage'], year_2022['education_real_wage']]

x = np.arange(len(sectors_2022))
width = 0.35
bars1 = axes[0].bar(x - width/2, wage_2022, width, label='Nominal Wage Growth',
                     color=[SECTOR_COLORS[s] for s in sectors_2022], edgecolor='white')
bars2 = axes[0].bar(x + width/2, real_2022, width, label='Real Wage Growth (after inflation)',
                     color='#E74C3C', edgecolor='white', alpha=0.85)

axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].axhline(year_2022['cpi_annual_pct'], color='black', linewidth=2,
                linestyle='-', label=f'CPI Inflation = {year_2022["cpi_annual_pct"]}%')
axes[0].set_title('2022: The Year Inflation Hit Hardest\nNominal vs Real Wage Growth',
                   fontweight='bold', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(sectors_2022)
axes[0].set_ylabel('% Change')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
axes[0].legend(fontsize=9)

for bar, val in zip(bars2, real_2022):
    axes[0].text(bar.get_x() + bar.get_width()/2, val - 0.3,
                 f'{val:.1f}%', ha='center', va='top', fontsize=10, fontweight='bold', color='white')

# Right: wage gap over 2019-2024
axes[1].plot(crisis['year'], crisis['cpi_annual_pct'], 'k--', linewidth=2.5,
             label='CPI Inflation', zorder=6)
for sector, col in [('Retail', 'retail_wage_growth'), ('Healthcare', 'healthcare_wage_growth'),
                     ('Tech', 'tech_wage_growth'), ('Education', 'education_wage_growth')]:
    axes[1].plot(crisis['year'], crisis[col], 'o-', color=SECTOR_COLORS[sector],
                 linewidth=2, markersize=7, label=sector)

axes[1].fill_between(crisis['year'], crisis['cpi_annual_pct'],
                      crisis['retail_wage_growth'], alpha=0.08, color='red')
axes[1].set_title('Wage vs Inflation Gap (2019–2024)\nShaded = where retail workers fell behind',
                   fontweight='bold', fontsize=12)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Annual % Change')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
axes[1].legend(fontsize=9)

plt.suptitle('The 2021–2023 Inflation Crisis — Impact on Real Wages', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('04_inflation_crisis_2022.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 7. Purchasing Power — What $50,000 in 2000 Is Worth Today

In [ ]:
base_salary = 50000

df['salary_needed'] = base_salary * (df['cum_cpi'] / 100)
df['retail_actual'] = base_salary * (df['cum_retail'] / 100)
df['healthcare_actual'] = base_salary * (df['cum_healthcare'] / 100)
df['tech_actual'] = base_salary * (df['cum_tech'] / 100)
df['education_actual'] = base_salary * (df['cum_education'] / 100)

fig, ax = plt.subplots(figsize=(15, 7))

ax.fill_between(df['year'], df['salary_needed'], base_salary, alpha=0.1, color='gray')
ax.plot(df['year'], df['salary_needed'], 'k--', linewidth=3,
        label='Salary needed to match inflation', zorder=6)

for sector, col in [('Retail', 'retail_actual'), ('Healthcare', 'healthcare_actual'),
                     ('Tech', 'tech_actual'), ('Education', 'education_actual')]:
    ax.plot(df['year'], df[col], 'o-', color=SECTOR_COLORS[sector],
            linewidth=2.5, markersize=6, label=f'{sector} actual salary')
    ax.annotate(f"${df[col].iloc[-1]:,.0f}",
                (df['year'].iloc[-1], df[col].iloc[-1]),
                textcoords='offset points', xytext=(8, 0),
                fontsize=9, color=SECTOR_COLORS[sector], fontweight='bold')

ax.annotate(f"${df['salary_needed'].iloc[-1]:,.0f}",
            (df['year'].iloc[-1], df['salary_needed'].iloc[-1]),
            textcoords='offset points', xytext=(8, 0),
            fontsize=9, color='black', fontweight='bold')

ax.axhline(base_salary, color='gray', linewidth=0.8, linestyle=':')
ax.text(2000.3, base_salary + 500, f'${base_salary:,} baseline (2000)', fontsize=8, color='gray')

ax.set_title(f'What Happened to a ${base_salary:,} Salary Since 2000?\n'
             'Nominal salary growth by sector vs what you needed to keep up with inflation',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Nominal Salary ($)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=10, loc='upper left')
ax.set_xticks(df['year'][::2])

plt.tight_layout()
plt.savefig('05_purchasing_power.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nIf you earned $50,000 in 2000, by 2026 you needed ${df["salary_needed"].iloc[-1]:,.0f} to match inflation.')
for sector, col in [('Retail', 'retail_actual'), ('Healthcare', 'healthcare_actual'),
                     ('Tech', 'tech_actual'), ('Education', 'education_actual')]:
    actual = df[col].iloc[-1]
    gap = actual - df['salary_needed'].iloc[-1]
    print(f'{sector}: ${actual:,.0f} — {"ahead by" if gap > 0 else "behind by"} ${abs(gap):,.0f}')

## 8. Heatmap — Which Years Were Good or Bad for Each Sector?

In [ ]:
heatmap_data = df[['year', 'retail_real_wage', 'healthcare_real_wage',
                    'tech_real_wage', 'education_real_wage']].copy()
heatmap_data = heatmap_data.set_index('year')
heatmap_data.columns = ['Retail', 'Healthcare', 'Tech', 'Education']
heatmap_data = heatmap_data.T

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(heatmap_data, cmap='RdYlGn', center=0, annot=True, fmt='.1f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Real Wage Growth (%)'},
            annot_kws={'size': 8})

ax.set_title('Real Wage Growth Heatmap by Sector (2000–2026)\n'
             'Green = wages beat inflation | Red = inflation beat wages',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Sector', fontsize=11)

plt.tight_layout()
plt.savefig('06_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 9. Winner & Loser Summary — Which Sector Won Over 26 Years?

In [ ]:
summary = {
    'Sector': ['Retail', 'Healthcare', 'Tech', 'Education'],
    'Cumulative Wage Growth (%)': [
        round(df['cum_retail'].iloc[-1] - 100, 1),
        round(df['cum_healthcare'].iloc[-1] - 100, 1),
        round(df['cum_tech'].iloc[-1] - 100, 1),
        round(df['cum_education'].iloc[-1] - 100, 1)
    ],
    'Years Real Wages Fell': [
        len(df[df['retail_real_wage'] < 0]),
        len(df[df['healthcare_real_wage'] < 0]),
        len(df[df['tech_real_wage'] < 0]),
        len(df[df['education_real_wage'] < 0])
    ]
}
summary_df = pd.DataFrame(summary)
cpi_total = round(df['cum_cpi'].iloc[-1] - 100, 1)
summary_df['Gap vs Inflation (%)'] = summary_df['Cumulative Wage Growth (%)'] - cpi_total
summary_df['Beat Inflation?'] = summary_df['Gap vs Inflation (%)'].apply(
    lambda x: '✅ Yes' if x > 0 else '❌ No')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cumulative growth bar chart
colors = [SECTOR_COLORS[s] for s in summary_df['Sector']]
bars = axes[0].bar(summary_df['Sector'], summary_df['Cumulative Wage Growth (%)'],
                    color=colors, edgecolor='white', linewidth=1)
axes[0].axhline(cpi_total, color='black', linewidth=2.5, linestyle='--',
                label=f'Total Inflation = {cpi_total}%')
axes[0].set_title('Cumulative Wage Growth vs Total Inflation\n(2000–2026)',
                   fontweight='bold', fontsize=12)
axes[0].set_ylabel('Cumulative Growth (%)')
axes[0].legend(fontsize=10)
for bar, val in zip(bars, summary_df['Cumulative Wage Growth (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1,
                 f'{val:.0f}%', ha='center', fontsize=11, fontweight='bold')

# Gap vs inflation
gap_colors = ['#2ECC71' if v > 0 else '#E74C3C' for v in summary_df['Gap vs Inflation (%)']]
bars2 = axes[1].bar(summary_df['Sector'], summary_df['Gap vs Inflation (%)'],
                     color=gap_colors, edgecolor='white', linewidth=1)
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_title('Gap vs Inflation Over 26 Years\nGreen = beat inflation | Red = lost to inflation',
                   fontweight='bold', fontsize=12)
axes[1].set_ylabel('Percentage Points Ahead/Behind Inflation')
for bar, val in zip(bars2, summary_df['Gap vs Inflation (%)']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + (1 if val >= 0 else -2),
                 f'{val:+.0f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('26-Year Scorecard: Which Sectors Beat Inflation? (2000–2026)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('07_sector_scorecard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n26-YEAR SCORECARD:')
print(summary_df.to_string(index=False))
print(f'\nTotal CPI inflation 2000–2026: {cpi_total}%')

## 10. Key Findings & Conclusions

In [ ]:
cpi_total = round(df['cum_cpi'].iloc[-1] - 100, 1)

print('=' * 65)
print('💸 US WAGES VS INFLATION — KEY FINDINGS (2000–2026)')
print('=' * 65)
print(f'\n📈 Total inflation 2000–2026:        {cpi_total}%')
print(f'\n🏆 Best performing sector:   Tech')
print(f'   Tech wages grew {df["cum_tech"].iloc[-1]-100:.0f}% — well ahead of inflation')
print(f'\n📉 Worst performing sector:  Education')
print(f'   Education wages grew {df["cum_education"].iloc[-1]-100:.0f}% — barely keeping pace')
print(f'\n🔴 Worst single year: 2022')
print(f'   CPI hit 8.0% while retail workers got just 5.1% raises')
print(f'   = a 2.9% real wage loss in a single year for retail workers')
print(f'\n💡 Key insight:')
print(f'   The 2021–2023 inflation spike erased years of real wage gains')
print(f'   for retail and education workers — the lowest paid sectors.')
print(f'   Tech workers, already earning more, stayed ahead throughout.')
print(f'   This widened the income inequality gap between sectors.')
print(f'\n🏦 Risk Management angle:')
print(f'   Workers with negative real wage growth face higher financial stress,')
print(f'   increased credit risk, and higher susceptibility to fraud schemes —')
print(f'   directly relevant to AML and consumer risk analytics.')
print('=' * 65)